# 🏥 VisionXAI — EfficientNet-B3 Diabetic Retinopathy Classifier
### Production Multi-Dataset Deep Learning Pipeline with Resume & Checkpointing

This notebook trains a genuine 5-class **EfficientNet-B3** model using **transfer learning**, **class-weighted loss**, **data augmentation**, and **automatic epoch checkpointing** so you can stop and resume anytime without losing progress.

--- 
### ⚡ Quick Kaggle Setup:
1. **Accelerator**: Select **GPU P100** or **GPU T4 x2** (right sidebar under *Session options*).
2. **Add Data**: Click **+ Add Data** (top right) → search and add:
   - `sovitrath/diabetic-retinopathy-224x224-2019-data`
3. Click **Run All**!

In [ ]:
# Cell 1: Environment Setup & GPU Diagnostic
!pip install timm pytorch-grad-cam onnx onnxruntime -q

import os, glob, time, json, shutil
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
import timm
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, cohen_kappa_score, confusion_matrix

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'🖥️ Device in use: {device}')
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f'🚀 GPU Name: {props.name} | VRAM: {props.total_memory / (1024**3):.2f} GB')

torch.manual_seed(42)
np.random.seed(42)

# Checkpoint resume configuration
RESUME_FROM_CHECKPOINT = True
CHECKPOINT_DIR = '/kaggle/working/checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print('✅ Environment initialized with automatic checkpointing.')

In [ ]:
# Cell 2: Robust Dataset Discovery & Metadata Ingestion
possible_roots = [
    '/kaggle/input/diabetic-retinopathy-224x224-2019-data',
    '/kaggle/input/aptos2019-blindness-detection',
    '/kaggle/input'
]

csv_files = []
for r in possible_roots:
    csv_files.extend(glob.glob(f'{r}/**/*.csv', recursive=True))

print('Discovered CSV metadata files:', csv_files)

data_records = []
csv_found = False
for c in csv_files:
    try:
        df = pd.read_csv(c)
        col_id = next((col for col in ['id_code', 'image', 'Image', 'id'] if col in df.columns), None)
        col_lbl = next((col for col in ['diagnosis', 'level', 'DR_grade', 'grade'] if col in df.columns), None)
        if col_id and col_lbl:
            print(f'✅ Found valid labels in: {c} (ID: {col_id}, Label: {col_lbl})')
            base_dir = os.path.dirname(c)
            for _, row in df.iterrows():
                img_name = str(row[col_id])
                if not img_name.endswith('.png') and not img_name.endswith('.jpg'):
                    # check both png and jpg
                    matches = glob.glob(f'{base_dir}/**/{img_name}.*', recursive=True)
                else:
                    matches = glob.glob(f'{base_dir}/**/{img_name}', recursive=True)
                if matches:
                    data_records.append({'image_path': matches[0], 'label': int(row[col_lbl])})
            if len(data_records) > 100:
                csv_found = True
                break
    except Exception as e:
        pass

# Fallback: Directory-based labels if folder names are 0, 1, 2, 3, 4
if not csv_found or len(data_records) < 100:
    print('Scanning for directory-based grade folders...')
    for grade in range(5):
        for ext in ['*.png', '*.jpg', '*.jpeg']:
            for f in glob.glob(f'/kaggle/input/**/{grade}/{ext}', recursive=True):
                data_records.append({'image_path': f, 'label': grade})

master_df = pd.DataFrame(data_records).drop_duplicates(subset=['image_path'])
print(f'📊 Total Retinal Images Ingested: {len(master_df)}')
print(master_df['label'].value_counts().sort_index())
assert len(master_df) > 100, 'Error: Could not locate sufficient retinal images. Check Kaggle input dataset.'

In [ ]:
# Cell 3: Stratified Dataset Split & Class-Weight Balancing
train_df, val_df = train_test_split(
    master_df, test_size=0.15, random_state=42, stratify=master_df['label']
)
print(f'Train samples: {len(train_df)} | Validation samples: {len(val_df)}')

# Compute inverse class frequencies to handle severe medical imbalance
class_counts = train_df['label'].value_counts().sort_index().values
class_weights = 1.0 / (class_counts + 1e-5)
class_weights = class_weights / class_weights.sum() * 5.0
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)
print('Computed Class Weights for Loss Function:', class_weights)

In [ ]:
# Cell 4: PyTorch Dataset & High-Fidelity Augmentation Pipeline
class RetinalDRDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        
    def __len__(self):
        return len(self.df)
        
    def __getitem__(self, idx):
        p = self.df.iloc[idx]['image_path']
        lbl = self.df.iloc[idx]['label']
        img = Image.open(p).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, torch.tensor(lbl, dtype=torch.long)

train_transforms = transforms.Compose([
    transforms.Resize((380, 380)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=25),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((380, 380)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_ds = RetinalDRDataset(train_df, transform=train_transforms)
val_ds = RetinalDRDataset(val_df, transform=val_transforms)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=16, shuffle=False, num_workers=2, pin_memory=True)
print('DataLoaders ready!')

In [ ]:
# Cell 5: EfficientNet-B3 Model Architecture with CAM Feature Forwarding
class VisionXAIClassifier(nn.Module):
    def __init__(self, num_classes=5):
        super().__init__()
        self.backbone = timm.create_model('efficientnet_b3', pretrained=True, num_classes=0)
        num_feats = self.backbone.num_features
        self.dropout = nn.Dropout(p=0.35)
        self.classifier = nn.Linear(num_feats, num_classes)
        
    def forward(self, x):
        feats = self.backbone(x)
        logits = self.classifier(self.dropout(feats))
        return logits
        
    def forward_with_features(self, x):
        feat_maps = self.backbone.forward_features(x)
        pooled = self.backbone.global_pool(feat_maps)
        if isinstance(pooled, tuple):
            pooled = pooled[0]
        logits = self.classifier(self.dropout(pooled))
        return logits, feat_maps

model = VisionXAIClassifier(num_classes=5).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=8, T_mult=2)
print('Model architecture instantiated with 5-class medical head.')

In [ ]:
# Cell 6: Checkpoint Ingestion & Resumption Engine
start_epoch = 0
best_val_kappa = -1.0
best_val_acc = 0.0

if RESUME_FROM_CHECKPOINT:
    existing_ckpts = sorted(glob.glob(f'{CHECKPOINT_DIR}/checkpoint_epoch_*.pth'))
    if existing_ckpts:
        latest_ckpt = existing_ckpts[-1]
        print(f'🔁 Loading existing checkpoint: {latest_ckpt}')
        ckpt_data = torch.load(latest_ckpt, map_location=device)
        model.load_state_dict(ckpt_data['model_state'])
        optimizer.load_state_dict(ckpt_data['opt_state'])
        scheduler.load_state_dict(ckpt_data['sched_state'])
        start_epoch = ckpt_data['epoch'] + 1
        best_val_kappa = ckpt_data.get('best_kappa', -1.0)
        best_val_acc = ckpt_data.get('best_acc', 0.0)
        print(f'✅ Successfully resumed! Training will continue from Epoch {start_epoch+1}')
    else:
        print('ℹ️ No previous checkpoint found. Starting fresh training session.')

In [ ]:
# Cell 7: Full Multi-Epoch Training Loop with Quadratic Weighted Kappa
TOTAL_EPOCHS = 25

for epoch in range(start_epoch, TOTAL_EPOCHS):
    model.train()
    running_loss, correct_train, total_train = 0.0, 0, 0
    
    for imgs, targets in train_loader:
        imgs, targets = imgs.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * imgs.size(0)
        preds = outputs.argmax(dim=1)
        correct_train += (preds == targets).sum().item()
        total_train += imgs.size(0)
        
    scheduler.step()
    train_loss = running_loss / total_train
    train_acc = correct_train / total_train
    
    # Validation phase
    model.eval()
    val_loss, correct_val, total_val = 0.0, 0, 0
    all_val_preds, all_val_targets = [], []
    
    with torch.no_grad():
        for imgs, targets in val_loader:
            imgs, targets = imgs.to(device), targets.to(device)
            outputs = model(imgs)
            loss = criterion(outputs, targets)
            val_loss += loss.item() * imgs.size(0)
            preds = outputs.argmax(dim=1)
            correct_val += (preds == targets).sum().item()
            total_val += imgs.size(0)
            all_val_preds.extend(preds.cpu().numpy())
            all_val_targets.extend(targets.cpu().numpy())
            
    val_acc = correct_val / total_val
    val_kappa = cohen_kappa_score(all_val_targets, all_val_preds, weights='quadratic')
    
    print(f'Epoch [{epoch+1}/{TOTAL_EPOCHS}] | Train Loss: {train_loss:.4f} Acc: {train_acc*100:.1f}% | Val Acc: {val_acc*100:.1f}% Kappa: {val_kappa:.4f}')
    
    # Save checkpoint every epoch so user can resume anytime
    ckpt_path = f'{CHECKPOINT_DIR}/checkpoint_epoch_{epoch+1:02d}.pth'
    torch.save({
        'epoch': epoch,
        'model_state': model.state_dict(),
        'opt_state': optimizer.state_dict(),
        'sched_state': scheduler.state_dict(),
        'val_acc': val_acc,
        'val_kappa': val_kappa,
        'best_kappa': max(best_val_kappa, val_kappa),
        'best_acc': max(best_val_acc, val_acc)
    }, ckpt_path)
    
    if val_kappa > best_val_kappa:
        best_val_kappa = val_kappa
        torch.save(model.state_dict(), '/kaggle/working/best_dr_model.pth')
        print(f'⭐ New Best Model Saved (Quadratic Kappa: {best_val_kappa:.4f})')

print('🎉 Training sequence completed!')

In [ ]:
# Cell 8: Comprehensive Clinical Metric Validation & Confusion Matrix
if os.path.exists('/kaggle/working/best_dr_model.pth'):
    model.load_state_dict(torch.load('/kaggle/working/best_dr_model.pth'))
    
model.eval()
y_true, y_pred = [], []
with torch.no_grad():
    for imgs, targets in val_loader:
        imgs = imgs.to(device)
        preds = model(imgs).argmax(dim=1)
        y_true.extend(targets.numpy())
        y_pred.extend(preds.cpu().numpy())

grade_labels = ['Grade 0 (No DR)', 'Grade 1 (Mild)', 'Grade 2 (Moderate)', 'Grade 3 (Severe)', 'Grade 4 (PDR)']
print(classification_report(y_true, y_pred, target_names=grade_labels, zero_division=0))
cm = confusion_matrix(y_true, y_pred)
print('Confusion Matrix:\n', cm)

In [ ]:
# Cell 9: ONNX Model Export & Inference Verification
model.eval()
dummy_input = torch.randn(1, 3, 380, 380).to(device)
export_onnx_path = '/kaggle/working/dr_model.onnx'

torch.onnx.export(
    model,
    dummy_input,
    export_onnx_path,
    input_names=['input_retina'],
    output_names=['logits'],
    dynamic_axes={'input_retina': {0: 'batch'}, 'logits': {0: 'batch'}},
    opset_version=14,
    do_constant_folding=True
)

# Export CAM model for heatmaps
class ModelWithCAM(nn.Module):
    def __init__(self, base_model):
        super().__init__()
        self.base = base_model
    def forward(self, x):
        return self.base.forward_with_features(x)

cam_model = ModelWithCAM(model)
cam_onnx_path = '/kaggle/working/dr_model_with_cam.onnx'
try:
    torch.onnx.export(
        cam_model,
        dummy_input,
        cam_onnx_path,
        input_names=['input_retina'],
        output_names=['logits', 'features'],
        dynamic_axes={'input_retina': {0: 'batch'}, 'logits': {0: 'batch'}, 'features': {0: 'batch'}},
        opset_version=14,
        do_constant_folding=True
    )
    print('✅ dr_model_with_cam.onnx exported successfully!')
except Exception as e:
    print(f'CAM ONNX export warning: {e}')

# Save classifier weights for Grad-CAM
fc_w = model.classifier.weight.detach().cpu().numpy()
np.save('/kaggle/working/fc_weights.npy', fc_w)

sz = os.path.getsize(export_onnx_path) / (1024 * 1024)
print(f'✅ Exported dr_model.onnx size: {sz:.2f} MB')

# Verification with ONNX Runtime
import onnxruntime as ort
sess = ort.InferenceSession(export_onnx_path)
test_out = sess.run(None, {'input_retina': dummy_input.cpu().numpy()})[0]
print('✅ ONNX Runtime verified logits output shape:', test_out.shape)

## 📥 Deployment Instructions
After training completes:
1. Go to the **Output** tab in Kaggle (right sidebar).
2. Download `dr_model.onnx` and place it in your local project folder:
   `Explainable-AI/ai/models/dr_model.onnx`
3. Download `dr_model_with_cam.onnx` and `fc_weights.npy` and place them into `Explainable-AI/ai/models/`.
4. Done! Your local backend will automatically recognize and run your production model.